<a href="https://colab.research.google.com/github/malikabaguari/IronHack-Malika-Projects-and-Labs/blob/main/api_project_ev_chargingstations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import time
import pandas as pd
from pandas import json_normalize
import os
from unidecode import unidecode
from itertools import product
import numpy as np
import re
#from unidecode import unidecode
#from geopy.geocoders import Nominatim
#from geopy.extra.rate_limiter import RateLimiter

In [ ]:
!pip install unidecode

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# Load the API key from the .env file
from dotenv import load_dotenv


load_dotenv()                                          # Read the .env file
openchargemap_api_key = os.getenv("OPENCHARGEMAP_API_KEY")               # Get the value of OPENCHARGEMAP_API_KEY
if openchargemap_api_key:
    print("Key loaded!")
else:
    print("ERROR: Key not found. Check your '.env' file.")

Key loaded!


## Extracting data from the API - Open Charge Map

In [ ]:
# Step 1 - Endpoint (URL) -
ev_url = "https://api.openchargemap.io/v3/poi"

#Step 2 -Importing the API key
headers = {
    "X-API-Key": openchargemap_api_key
}

#Step 3 - Defining the countries to download data
countries = ["ES","NL","DE","PT","FR"]

#Step 4 - creating an empty list to store data from all countries
all_data = []

# Step 5 - Looping through each country and setting the API request parameters for each country, including the output format, country code, and maximum number of charging station records to retrieve.
for country in countries:

    params = {
        "output": "json",
        "countrycode": country,
        "maxresults": 30000
    }

    response = requests.get(
        ev_url,
        headers=headers,
        params=params
    )

#Step 6 - Checking if the request was successful
    if response.status_code == 200:

        data = response.json()

        print(f"{country}: {len(data)} stations downloaded")

        # Add country code to each record
        for station in data:
            station["CountryCode"] = country

        # Add data to master list
        all_data.extend(data)

        # Pause between requests
        time.sleep(1)

    else:
        print(f"Error downloading data for {country}")



ES: 20142 stations downloaded
NL: 8160 stations downloaded
DE: 24589 stations downloaded
PT: 3736 stations downloaded
FR: 16088 stations downloaded


## Normalizing and Analysing the data

In [ ]:
raw_ev_df = pd.json_normalize(all_data)

In [ ]:
raw_ev_df.sample()

,UserComments,PercentageSimilarity,MediaItems,IsRecentlyVerified,DateLastVerified,ID,UUID,ParentChargePointID,DataProviderID,DataProvidersReference,...,AddressInfo.Longitude,AddressInfo.ContactTelephone1,AddressInfo.ContactTelephone2,AddressInfo.ContactEmail,AddressInfo.AccessComments,AddressInfo.RelatedURL,AddressInfo.Distance,AddressInfo.DistanceUnit,UsageType,OperatorInfo
16273,None,None,None,False,2022-09-21T08:33:00Z,203731,96D5F6D1-4E0C-4FE6-B642-40B51925194A,None,1,None,...,0.512685,None,None,None,None,None,None,0,NaN,NaN


In [ ]:
#Check for duplicates
print("Total rows:", len(raw_ev_df))
print("Unique stations:", raw_ev_df["ID"].nunique())

Total rows: 72715
Unique stations: 72715


In [ ]:
## Identifying the EV stations records were created and focussing from 2020 to 2024

In [ ]:
# Convert DateCreated to datetime
raw_ev_df["DateCreated"] = pd.to_datetime(
    raw_ev_df["DateCreated"],
    errors="coerce"
)

# Extract the year
raw_ev_df["Station_Created_Year"] = raw_ev_df["DateCreated"].dt.year

# Countries to analyse
countries = ["ES", "NL", "DE", "PT", "FR"]

# Filter the dataframe
filtered_df = raw_ev_df[
    raw_ev_df["CountryCode"].isin(countries)
].copy()

# Count unique charging stations added per year for each country
stations_by_year_country = (
    filtered_df
    .groupby(["CountryCode", "Station_Created_Year"])["ID"]
    .nunique()
    .reset_index(name="Number_of_Stations_Added")
)

stations_by_year_country

,CountryCode,Station_Created_Year,Number_of_Stations_Added
0,DE,2011,13
1,DE,2012,8
2,DE,2013,46
3,DE,2014,53
4,DE,2015,4351
...,...,...,...
73,PT,2022,400
74,PT,2023,1148
75,PT,2024,18
76,PT,2025,415


In [ ]:
# Create a pivot table showing the number of EV charging stations added each year for each country.
stations_by_year_country_pivot = (
    stations_by_year_country
    .pivot(
        index="Station_Created_Year",
        columns="CountryCode",
        values="Number_of_Stations_Added"
    )
    .fillna(0)
    .astype(int)
)

stations_by_year_country_pivot

CountryCode,DE,ES,FR,NL,PT
Station_Created_Year,,,,,
2011,13,2,127,5,97
2012,8,0,41,26,82
2013,46,0,650,29,2
2014,53,79,125,58,134
2015,4351,64,41,5322,16
2016,2466,303,183,2051,13
2017,2897,208,440,81,76
2018,1996,226,227,48,104
2019,399,149,192,96,173


In [ ]:
# Checking the columns in the raw df
raw_ev_df.columns

Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'CountryCode', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
   

In [ ]:
#Creating a copy of the dataframe
working_ev_df = raw_ev_df.copy()

In [ ]:
# Display the first five rows of the dataset to preview the data and verify that it has been loaded correctly.
working_ev_df.head()

,UserComments,PercentageSimilarity,MediaItems,IsRecentlyVerified,DateLastVerified,ID,UUID,ParentChargePointID,DataProviderID,DataProvidersReference,...,AddressInfo.ContactTelephone1,AddressInfo.ContactTelephone2,AddressInfo.ContactEmail,AddressInfo.AccessComments,AddressInfo.RelatedURL,AddressInfo.Distance,AddressInfo.DistanceUnit,UsageType,OperatorInfo,Station_Created_Year
0,None,None,None,True,2026-06-23T12:47:00Z,495071,3737F5E4-4AF8-40D6-A872-F460219AD236,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
1,None,None,None,True,2026-06-23T12:41:00Z,495070,D5910A29-005C-43DE-A1A6-39338B9CC98F,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
2,None,None,None,True,2026-06-23T12:37:00Z,495069,43505BA7-1535-41AD-8154-27F675A38DD6,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
3,None,None,None,True,2026-06-23T12:32:00Z,495068,C903E57E-E0D1-42F8-8896-1A546226589C,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026
4,None,None,None,True,2026-06-23T12:25:00Z,495067,07D026C7-D363-483E-83BF-79B9DED47F66,None,1,None,...,None,None,None,None,None,None,0,NaN,NaN,2026


In [ ]:
# Count the number of missing (null) values in each column to identify incomplete data.
working_ev_df.isnull().sum()

UserComments                70921
PercentageSimilarity        72715
MediaItems                  62875
IsRecentlyVerified              0
DateLastVerified                0
                            ...  
AddressInfo.Distance        72715
AddressInfo.DistanceUnit        0
UsageType                   72715
OperatorInfo                72715
Station_Created_Year            0
Length: 86, dtype: int64

In [ ]:
# Count the number of records for each country to check the distribution of charging stations by country.
working_ev_df["CountryCode"].value_counts()

CountryCode
DE    24589
ES    20142
FR    16088
NL     8160
PT     3736
Name: count, dtype: int64

In [ ]:
# Count the number of operational and non-operational charging stations to assess their current status.
working_ev_df["StatusType.IsOperational"].value_counts()

StatusType.IsOperational
True     68483
False     1575
Name: count, dtype: int64

In [ ]:
# Display the top 20 towns with the highest number of charging station records.
working_ev_df["AddressInfo.Town"].value_counts().head(20)

AddressInfo.Town
Berlin        854
Hamburg       621
München       508
Madrid        451
Den Haag      434
Amsterdam     420
Rotterdam     349
              293
Barcelona     282
Stuttgart     275
Essen         254
Lisboa        230
Utrecht       213
Hannover      198
Paris         154
Düsseldorf    150
Bremen        139
Leipzig       136
Eindhoven     129
Köln          126
Name: count, dtype: int64

In [ ]:
# Count the number of completely duplicated rows in the dataset.
working_ev_df["ID"].duplicated().sum()

np.int64(0)

## Dropping unnecessary columns

In [ ]:
working_ev_df.columns


Index(['UserComments', 'PercentageSimilarity', 'MediaItems',
       'IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID',
       'ParentChargePointID', 'DataProviderID', 'DataProvidersReference',
       'OperatorID', 'OperatorsReference', 'UsageTypeID', 'UsageCost',
       'Connections', 'NumberOfPoints', 'GeneralComments', 'DatePlanned',
       'DateLastConfirmed', 'StatusTypeID', 'DateLastStatusUpdate',
       'MetadataValues', 'DataQualityLevel', 'DateCreated',
       'SubmissionStatusTypeID', 'CountryCode', 'DataProvider.WebsiteURL',
       'DataProvider.Comments',
       'DataProvider.DataProviderStatusType.IsProviderEnabled',
       'DataProvider.DataProviderStatusType.ID',
       'DataProvider.DataProviderStatusType.Title',
       'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
       'DataProvider.IsApprovedImport', 'DataProvider.License',
       'DataProvider.DateLastImported', 'DataProvider.ID',
       'DataProvider.Title', 'OperatorInfo.WebsiteURL',
   

In [ ]:
#Removing Columns that are not relevant to our analysis
cols_to_drop = ['UserComments', 'PercentageSimilarity', 'MediaItems','DateLastVerified', 'UUID','ParentChargePointID',
    'DataProviderID', 'DataProvidersReference', 'OperatorID', 'OperatorsReference',
    'GeneralComments', 'DatePlanned', 'DateLastConfirmed', 'DateLastStatusUpdate',
    'MetadataValues', 'DataQualityLevel',  'SubmissionStatusTypeID',
    'DataProvider.WebsiteURL', 'DataProvider.Comments',
    'DataProvider.DataProviderStatusType.IsProviderEnabled',
    'DataProvider.DataProviderStatusType.ID',
    'DataProvider.DataProviderStatusType.Title',
    'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed',
    'DataProvider.IsApprovedImport', 'DataProvider.License',
    'DataProvider.DateLastImported', 'DataProvider.ID',
    'DataProvider.Title', 'OperatorInfo.WebsiteURL',
    'OperatorInfo.Comments', 'OperatorInfo.PhonePrimaryContact',
    'OperatorInfo.PhoneSecondaryContact', 'OperatorInfo.IsPrivateIndividual',
    'OperatorInfo.AddressInfo', 'OperatorInfo.BookingURL',
    'OperatorInfo.ContactEmail', 'OperatorInfo.FaultReportEmail',
    'OperatorInfo.IsRestrictedEdit', 'OperatorInfo.ID',
    'OperatorInfo.Title', 'UsageType.IsPayAtLocation',
    'StatusType.IsUserSelectable', 'StatusType.ID',
    'SubmissionStatus.IsLive', 'SubmissionStatus.ID',
    'SubmissionStatus.Title', 'AddressInfo.ID',
    'AddressInfo.AddressLine2', 'AddressInfo.Postcode',
    'AddressInfo.CountryID', 'AddressInfo.Country.ISOCode',
    'AddressInfo.Country.ContinentCode', 'AddressInfo.Country.ID',
    'AddressInfo.ContactTelephone1', 'AddressInfo.ContactTelephone2',
    'AddressInfo.ContactEmail', 'AddressInfo.AccessComments',
    'AddressInfo.RelatedURL', 'AddressInfo.Distance',
    'AddressInfo.DistanceUnit'
]

working_ev_df = working_ev_df.drop(columns=cols_to_drop, errors='ignore')

working_ev_df.head()

,IsRecentlyVerified,ID,UsageTypeID,UsageCost,Connections,NumberOfPoints,StatusTypeID,DateCreated,CountryCode,UsageType.IsMembershipRequired,...,AddressInfo.Title,AddressInfo.AddressLine1,AddressInfo.Town,AddressInfo.StateOrProvince,AddressInfo.Country.Title,AddressInfo.Latitude,AddressInfo.Longitude,UsageType,OperatorInfo,Station_Created_Year
0,True,495071,4.0,"0,54€/kWh DC","[{'ID': 842056, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:45:00+00:00,ES,True,...,E.S. Q8 Gavá,Carrer de l'Enginy,Gavà,Catalunya,Spain,41.302207,2.016679,NaN,NaN,2026
1,True,495070,4.0,"0,60€/kWh","[{'ID': 842054, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:40:00+00:00,ES,True,...,ALDI Torroella,GI-641,Torroella de Montgrí,Catalunya,Spain,42.041647,3.135979,NaN,NaN,2026
2,True,495069,4.0,"0,54€/kWh DC","[{'ID': 842053, 'ConnectionTypeID': 33, 'Conne...",2.0,50,2026-06-23 12:35:00+00:00,ES,True,...,E.S. Q8 Ripoll,Carrer dels Martinets,Ripoll,Catalunya,Spain,42.215653,2.168983,NaN,NaN,2026
3,True,495068,4.0,"0,37€/kWh","[{'ID': 842052, 'ConnectionTypeID': 33, 'Conne...",4.0,50,2026-06-23 12:30:00+00:00,ES,True,...,MapisaOil,Eix Salou - Andorra C-14,None,Catalunya,Spain,42.317620,1.388048,NaN,NaN,2026
4,True,495067,4.0,"0,25€/kWh","[{'ID': 842051, 'ConnectionTypeID': 25, 'Conne...",4.0,50,2026-06-23 12:24:00+00:00,ES,True,...,Mercadona Cervera,Carrer de Montoliu,Cervera,Catalunya,Spain,41.672195,1.269770,NaN,NaN,2026


In [ ]:
# Check whether the 'DateCreated' column exists and list all columns containing the word 'Date'.
print("DateCreated exists:", "DateCreated" in working_ev_df.columns)

[col for col in working_ev_df.columns if "Date" in col]

DateCreated exists: True


['DateCreated']

In [ ]:
working_ev_df.columns

Index(['IsRecentlyVerified', 'ID', 'UsageTypeID', 'UsageCost', 'Connections',
       'NumberOfPoints', 'StatusTypeID', 'DateCreated', 'CountryCode',
       'UsageType.IsMembershipRequired', 'UsageType.IsAccessKeyRequired',
       'UsageType.ID', 'UsageType.Title', 'StatusType.IsOperational',
       'StatusType.Title', 'AddressInfo.Title', 'AddressInfo.AddressLine1',
       'AddressInfo.Town', 'AddressInfo.StateOrProvince',
       'AddressInfo.Country.Title', 'AddressInfo.Latitude',
       'AddressInfo.Longitude', 'UsageType', 'OperatorInfo',
       'Station_Created_Year'],
      dtype='object')

## Codes to arrive at Province and Autonomous Community

In [ ]:
!pip install unidecode

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# Create a function to standardize region names by cleaning text, removing duplicates and formatting inconsistencies, and mapping  different name variations to a single, consistent region name.
mapping = {
    "comunidad de madrid": "Madrid",
    "community of madrid": "Madrid",
    "madrid ": "Madrid",
    "madrid.": "Madrid",

    "catalunya": "Catalonia",
    "catalonia": "Catalonia",

    "comunitat valenciana": "Valencian Community",
    "valencian community": "Valencian Community",

    "euskadi": "Basque Country",
    "autonomous community of the basque country": "Basque Country",

    "illes balears": "Balearic Islands",
    "isles baleares": "Balearic Islands",
    "balearic islands": "Balearic Islands",

    "castilla y leon": "Castile and León",
    "castile and leon": "Castile and León",

    "castilla-la mancha": "Castile-La Mancha",
    "castile-la mancha": "Castile-La Mancha",

    "region of murcia": "Murcia",
    "región de murcia": "Murcia",

    "galicia": "Galicia",
    "galicia ": "Galicia",

    "asturias / asturies": "Asturias",
    "asturias": "Asturias",

    "navarra / nafarroa": "Navarra",
    "navarre": "Navarra",

    "la rioja": "La Rioja",
    "rioja": "La Rioja",

    "canarias": "Canary Islands",
    "canary islands": "Canary Islands",
}

def clean_state(value):
    """
    Cleans and standardizes region names by removing formatting inconsistencies,
    normalizing text, and mapping different name variations to a consistent name.
    """
    if pd.isna(value):
        return value

    # Convert to string
    v = str(value)

    # Remove extra spaces
    v = v.strip()

    # Normalize accents (Játiva → Jativa)
    v = unidecode(v)

    # Lowercase for matching
    v_low = v.lower()

    # Remove punctuation & double spaces
    v_low = re.sub(r"[^a-z0-9\s/]", "", v_low)
    v_low = re.sub(r"\s+", " ", v_low).strip()

    # Apply mapping if exists
    if v_low in mapping:
        return mapping[v_low]

    # Final formatting: Title Case
    return v.title()

In [ ]:
# Apply the cleaning function to the State/Province column to standardize all region names.
working_ev_df["AddressInfo.StateOrProvince"] = working_ev_df["AddressInfo.StateOrProvince"].apply(clean_state)

In [ ]:
# Display the top 20 most frequent region names after the data has been cleaned and standardized.
working_ev_df["AddressInfo.StateOrProvince"].value_counts().head(20)

AddressInfo.StateOrProvince
Catalonia              2740
Valencian Community    1963
Madrid                 1879
Andalucia              1529
Castile and León       1062
Andalusia               823
Balearic Islands        818
                        759
Galicia                 754
Basque Country          564
Aragon                  500
Castilla-La Mancha      441
Hamburg                 421
Berlin                  399
Lisboa                  398
Cantabria               373
Canary Islands          355
Extremadura             347
Region De Murcia        327
Bayern                  311
Name: count, dtype: int64

In [ ]:
# Install the geopy library, which is used to convert geographic coordinates (latitude and longitude) into location information such as provinces or cities.
!pip install geopy

Defaulting to user installation because normal site-packages is not writeable


## Following codes are converted to raw as these take hours to run. In this scenario, it would take around 12-14 hours.

## Merging Provinces to Automous Community

## Looking at the Highway perspective

In [ ]:
# Make a copy
ev_df = working_ev_df.copy()

# Convert DateCreated to datetime
ev_df["DateCreated"] = pd.to_datetime(ev_df["DateCreated"], errors="coerce")
ev_df["Station_Created_Year"] = ev_df["DateCreated"].dt.year

# Keep only 2020 to 2024
ev_2020_2024_df = ev_df[
    ev_df["Station_Created_Year"].between(2020, 2024)
].copy()

print("Rows from 2020-2024:", len(ev_2020_2024_df))

Rows from 2020-2024: 41639


In [ ]:
# Identify charging stations located along the France–Spain border corridor  using latitude and longitude, then count the total number of stations in this area.
ev_2020_2024_df["France_Spain_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(40.5, 42.9) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(0.5, 3.5)
)

print("France-Spain corridor stations:",
      ev_2020_2024_df["France_Spain_Corridor"].sum())

France-Spain corridor stations: 2747


In [ ]:
# Calculate the percentage of total charging stations that are located within the France–Spain corridor and display the result as a percentage.
total_stations = len(ev_2020_2024_df)

corridor_stations = ev_2020_2024_df["France_Spain_Corridor"].sum()

percentage = corridor_stations / total_stations * 100

print(f"{percentage:.1f}%")

6.6%


In [ ]:
# Identify charging stations located along the Portugal_Spain border corridor  using latitude and longitude, then count the total number of stations in this area.
ev_2020_2024_df["Portugal_Spain_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(37.0, 42.3) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-9.5, -5.5)
)

print("Portugal-Spain corridor stations:",
      ev_2020_2024_df["Portugal_Spain_Corridor"].sum())

Portugal-Spain corridor stations: 3861


In [ ]:
# Identify charging stations located along the Mediterranean_Spain border corridor  using latitude and longitude, then count the total number of stations in this area.
ev_2020_2024_df["Mediterranean_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(37.5, 42.5) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-1.0, 3.5)
)

print("Mediterranean corridor stations:",
      ev_2020_2024_df["Mediterranean_Corridor"].sum())

Mediterranean corridor stations: 5436


In [ ]:
# Identify charging stations located along the Madrid_Hub using latitude and longitude, then count the total number of stations in this area.
ev_2020_2024_df["Madrid_Hub"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(39.8, 41.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-4.5, -2.8)
)

print("Madrid hub stations:",
      ev_2020_2024_df["Madrid_Hub"].sum())

Madrid hub stations: 1954


In [ ]:
# Identify charging stations located along the Madrid_Valencia_Corridor  using latitude and longitude, then count the total number of stations in this area.
ev_2020_2024_df["Madrid_Valencia_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(38.8, 40.8) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-4.2, -0.2)
)

print("Madrid-Valencia corridor stations:",
      ev_2020_2024_df["Madrid_Valencia_Corridor"].sum())

Madrid-Valencia corridor stations: 2975


In [ ]:
# Identify charging stations located along the Madrid_Andalusia_Corridor  using latitude and longitude, then count the total number of stations in this area.
ev_2020_2024_df["Madrid_Andalusia_Corridor"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(36.3, 40.6) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-5.5, -2.8)
)

print("Madrid-Andalusia corridor stations:",
      ev_2020_2024_df["Madrid_Andalusia_Corridor"].sum())

Madrid-Andalusia corridor stations: 3117


In [ ]:
# Optional exploratory analysis:
# Create location flags for key tourism regions using latitude and longitude.
# This was used to explore whether tourist-heavy areas could be relevant for future EV charging station recommendations.

ev_2020_2024_df["Tourism_Barcelona_Catalonia"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(41.0, 42.5) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(1.5, 3.3)
)

ev_2020_2024_df["Tourism_Madrid"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(39.8, 41.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-4.5, -2.8)
)

ev_2020_2024_df["Tourism_Andalusia"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(36.0, 38.8) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-7.5, -1.5)
)

ev_2020_2024_df["Tourism_Valencia_Alicante"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(37.8, 40.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(-1.2, 0.8)
)

ev_2020_2024_df["Tourism_Balearic_Islands"] = (
    ev_2020_2024_df["AddressInfo.Latitude"].between(38.5, 40.2) &
    ev_2020_2024_df["AddressInfo.Longitude"].between(1.0, 4.5)
)

In [ ]:
# Summarize and compare the number of charging stations across different strategic corridors, transport routes, and tourism regions.

angle_columns = [
    "France_Spain_Corridor",
    "Portugal_Spain_Corridor",
    "Mediterranean_Corridor",
    "Madrid_Hub",
    "Madrid_Valencia_Corridor",
    "Madrid_Andalusia_Corridor",
    "Tourism_Barcelona_Catalonia",
    "Tourism_Madrid",
    "Tourism_Andalusia",
    "Tourism_Valencia_Alicante",
    "Tourism_Balearic_Islands"
]

summary = pd.DataFrame({
    "Angle": angle_columns,
    "Number_of_Stations": [ev_2020_2024_df[col].sum() for col in angle_columns]
})

summary = summary.sort_values("Number_of_Stations", ascending=False)

summary

,Angle,Number_of_Stations
2,Mediterranean_Corridor,5436
1,Portugal_Spain_Corridor,3861
5,Madrid_Andalusia_Corridor,3117
4,Madrid_Valencia_Corridor,2975
0,France_Spain_Corridor,2747
8,Tourism_Andalusia,2526
6,Tourism_Barcelona_Catalonia,2075
9,Tourism_Valencia_Alicante,2039
3,Madrid_Hub,1954
7,Tourism_Madrid,1954


In [ ]:
ev_2020_2024_df.to_csv("ev_spain_2020_2024_corridor_analysis.csv", index=False)

print("File saved successfully!")

File saved successfully!


## Extracting to excel

## Extracting data from the API - EUROSTAT API


In [ ]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=ES"

response = requests.get(url)
response.raise_for_status()
data = response.json()



def jsonstat_to_df(data):
    # Get the dimension names from the Eurostat data
    # Example: ["freq", "unit", "mot_nrg", "geo", "time"]
    dim_ids = data["id"]

    # This contains the category information for each dimension
    dimensions = data["dimension"]

    # These are the actual numbers from Eurostat
    values = data["value"]

    # This list will store all the possible codes for each dimension
    all_codes = []

    # Go through each dimension one by one
    for dim in dim_ids:
        category = dimensions[dim]["category"]

        # The index tells us the order of the codes
        index = category["index"]

        # Sometimes Eurostat gives the index as a dictionary
        # Example: {"ELC": 0, "PET": 1}
        if isinstance(index, dict):
            codes = sorted(index, key=index.get)

        # Sometimes Eurostat gives the index as a list
        # Example: ["ELC", "PET"]
        else:
            codes = index

        # Save the ordered codes for this dimension
        all_codes.append(codes)

    # This will become the final table
    rows = []

    # product(*all_codes) creates every possible combination
    # Example:
    # geo = ["ES"]
    # time = ["2023", "2024"]
    # mot_nrg = ["PET", "ELC"]
    #
    # It creates:
    # ES, 2023, PET
    # ES, 2023, ELC
    # ES, 2024, PET
    # ES, 2024, ELC
    for flat_index, codes_combo in enumerate(product(*all_codes)):

        # Eurostat can store values as a dictionary or as a list.
        # This handles both cases.
        if isinstance(values, dict):
            value = values.get(str(flat_index))
        else:
            value = values[flat_index]

        # Skip empty values
        if value is None:
            continue

        # This dictionary will become one row in the DataFrame
        row = {}

        # Match each dimension name with its code
        # Example:
        # dim = "mot_nrg"
        # code = "ELC"
        for dim, code in zip(dim_ids, codes_combo):

            # Get the readable label if Eurostat provides one
            # Example: "ELC" becomes "Electricity"
            label = dimensions[dim]["category"].get("label", {}).get(code, code)

            # Add the raw code column
            row[dim] = code

            # Add the readable label column
            row[f"{dim}_label"] = label

        # Add the actual Eurostat number
        row["value"] = value

        # Save this row
        rows.append(row)

    # Convert the list of rows into a pandas table
    return pd.DataFrame(rows)

    # Convert all rows into a pandas DataFrame
    return pd.DataFrame(rows)

df = jsonstat_to_df(data)

df.head()


,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512
2,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2015,2015,22355549
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
4,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2017,2017,24676557


In [ ]:
# Display all unique fuel type categories available in the dataset.
df["mot_nrg_label"].unique()

array(['Total', 'Petrol', 'Liquefied petroleum gases (LPG)', 'Diesel',
       'Natural gas', 'Electricity', 'Alternative energy',
       'Petrol (excluding hybrids) \xa0', 'Hybrid electric-petrol',
       'Plug-in hybrid petrol-electric \xa0',
       'Diesel (excluding hybrids) \xa0', 'Hybrid diesel-electric',
       'Plug-in hybrid diesel-electric \xa0',
       'Hydrogen and fuel cells\xa0', 'Bioethanol', 'Biodiesel',
       'Bi-fuel', 'Other'], dtype=object)

In [ ]:
# Remove diesel and petrol vehicle records to focus the analysis on electric and alternative fuel vehicle types.

rows_to_drop = df[
    df["mot_nrg_label"].isin(["Diesel", "Petrol"])
].index

df = df.drop(rows_to_drop)

df

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512
2,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2015,2015,22355549
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
4,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2017,2017,24676557
...,...,...,...,...,...,...,...,...,...,...,...
192,A,Annual,NR,Number,OTH,Other,ES,Spain,2020,2020,10814
193,A,Annual,NR,Number,OTH,Other,ES,Spain,2021,2021,11578
194,A,Annual,NR,Number,OTH,Other,ES,Spain,2022,2022,11420
195,A,Annual,NR,Number,OTH,Other,ES,Spain,2023,2023,11406


In [ ]:
#we just want to focuson EV vehicles so we will filter the data to only include rows where the "mot_nrg_label" column contains the word "electric"

electric_spain = df[df["mot_nrg_label"].str.contains("electric", case=False, na=False)]

electric_spain

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
56,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2013,2013,2021
57,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2014,2014,2832
58,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2015,2015,5243
59,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2016,2016,5756
60,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2017,2017,9681
61,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2018,2018,15865
62,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2019,2019,26197
63,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2020,2020,43203
64,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2021,2021,66085
65,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2022,2022,95617


In [ ]:
#looking just at 2024 data in spain

electric_spain_2024 = electric_spain[electric_spain["time"] == "2024"]
electric_spain_2024

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
67,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2024,2024,208240
100,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,ES,Spain,2024,2024,1383928
111,A,Annual,NR,Number,ELC_PET_PI,Plug-in hybrid petrol-electric,ES,Spain,2024,2024,226621
133,A,Annual,NR,Number,ELC_DIE_HYB,Hybrid diesel-electric,ES,Spain,2024,2024,192829
144,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,ES,Spain,2024,2024,13023


In [ ]:
# Retrieve passenger car data for Portugal from the Eurostat API, convert the JSON response into a DataFrame, and display the result.
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=PT&lang=en"

response = requests.get(url)
data_portugal = response.json()

df_portugal = jsonstat_to_df(data_portugal)

df_portugal

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2013,2013,4327478
1,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2014,2014,4699645
2,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2015,2015,4722963
3,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2016,2016,4850229
4,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2017,2017,5059472
...,...,...,...,...,...,...,...,...,...,...,...
209,A,Annual,NR,Number,OTH,Other,PT,Portugal,2020,2020,0
210,A,Annual,NR,Number,OTH,Other,PT,Portugal,2021,2021,1
211,A,Annual,NR,Number,OTH,Other,PT,Portugal,2022,2022,0
212,A,Annual,NR,Number,OTH,Other,PT,Portugal,2023,2023,0


In [ ]:
# Retrieve passenger car data for France from the Eurostat API, convert the JSON response into a DataFrame, and display the result.

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=FR&lang=en"

response = requests.get(url)
data_france = response.json()

df_france = jsonstat_to_df(data_france)

df_france

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,FR,France,2013,2013,36149230
1,A,Annual,NR,Number,TOTAL,Total,FR,France,2014,2014,36647883
2,A,Annual,NR,Number,TOTAL,Total,FR,France,2015,2015,37180101
3,A,Annual,NR,Number,TOTAL,Total,FR,France,2016,2016,37673476
4,A,Annual,NR,Number,TOTAL,Total,FR,France,2017,2017,38117663
...,...,...,...,...,...,...,...,...,...,...,...
211,A,Annual,NR,Number,OTH,Other,FR,France,2020,2020,2676
212,A,Annual,NR,Number,OTH,Other,FR,France,2021,2021,2625
213,A,Annual,NR,Number,OTH,Other,FR,France,2022,2022,2727
214,A,Annual,NR,Number,OTH,Other,FR,France,2023,2023,2534


In [ ]:
# Retrieve passenger car data for the Netherlands from the Eurostat API, convert the JSON response into a DataFrame, and display the result.

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=NL&lang=en"

response = requests.get(url)
data_netherlands = response.json()

df_netherlands = jsonstat_to_df(data_netherlands)

df_netherlands

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2013,2013,7932290
1,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2014,2014,7979083
2,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2015,2015,8100864
3,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2016,2016,8222974
4,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2017,2017,8373244
...,...,...,...,...,...,...,...,...,...,...,...
187,A,Annual,NR,Number,OTH,Other,NL,Netherlands,2020,2020,0
188,A,Annual,NR,Number,OTH,Other,NL,Netherlands,2021,2021,0
189,A,Annual,NR,Number,OTH,Other,NL,Netherlands,2022,2022,0
190,A,Annual,NR,Number,OTH,Other,NL,Netherlands,2023,2023,0


In [ ]:
# Retrieve passenger car data for Germany from the Eurostat API, convert the JSON response into a DataFrame, and store it for analysis.

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carpda?geo=DE&lang=en"

response = requests.get(url)
data_germany = response.json()

df_germany = jsonstat_to_df(data_germany)

In [ ]:
# Retrieve Portugal's passenger cars per inhabitant data (Passenger cars per 1,000 inhabitants) from the Eurostat API, convert the JSON response into a DataFrame, and preview the first few rows.

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carhab?geo=PT&lang=en"

response = requests.get(url)
data_carhab_pt = response.json()

df_carhab_pt = jsonstat_to_df(data_carhab_pt)

df_carhab_pt.head()

,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,PT,Portugal,1990,1990,256
1,A,Annual,NR,Number,PT,Portugal,1991,1991,279
2,A,Annual,NR,Number,PT,Portugal,1992,1992,306
3,A,Annual,NR,Number,PT,Portugal,1993,1993,330
4,A,Annual,NR,Number,PT,Portugal,1994,1994,353


In [ ]:
#merging the dataframes together to have a complete dataset for all the countries
df_total = pd.concat([df, df_portugal, df_france, df_netherlands, df_germany], ignore_index=True)
df_total

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
1,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2014,2014,22029512
2,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2015,2015,22355549
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830
4,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2017,2017,24676557
...,...,...,...,...,...,...,...,...,...,...,...
948,A,Annual,NR,Number,OTH,Other,DE,Germany,2020,2020,9514
949,A,Annual,NR,Number,OTH,Other,DE,Germany,2021,2021,9190
950,A,Annual,NR,Number,OTH,Other,DE,Germany,2022,2022,8812
951,A,Annual,NR,Number,OTH,Other,DE,Germany,2023,2023,8394


In [ ]:
# Display the number of records available for each country in the combined dataset.
df_total["geo"].value_counts()


geo
FR    216
PT    214
NL    192
ES    175
DE    156
Name: count, dtype: int64

In [ ]:
# Check each column for missing values in the combined dataset.
df_total.isna().sum()


freq             0
freq_label       0
unit             0
unit_label       0
mot_nrg          0
mot_nrg_label    0
geo              0
geo_label        0
time             0
time_label       0
value            0
dtype: int64

In [ ]:
df_total["time"] = df_total["time"].astype(int)
df_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   freq           953 non-null    object
 1   freq_label     953 non-null    object
 2   unit           953 non-null    object
 3   unit_label     953 non-null    object
 4   mot_nrg        953 non-null    object
 5   mot_nrg_label  953 non-null    object
 6   geo            953 non-null    object
 7   geo_label      953 non-null    object
 8   time           953 non-null    int64 
 9   time_label     953 non-null    object
 10  value          953 non-null    int64 
dtypes: int64(2), object(9)
memory usage: 82.0+ KB


In [ ]:
df_total.duplicated().sum()

np.int64(0)

In [ ]:
#filtering the data to only include rows where the "mot_nrg_label" column contains the word "electric"
df_total_ev= df_total[df_total["mot_nrg_label"].str.contains("electric", case=False, na=False)]
df_total_ev

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
34,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2013,2013,2021
35,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2014,2014,2832
36,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2015,2015,5243
37,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2016,2016,5756
38,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2017,2017,9681
...,...,...,...,...,...,...,...,...,...,...,...
924,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2020,2020,29846
925,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2021,2021,49822
926,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2022,2022,62162
927,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2023,2023,63918


In [ ]:
#looking just at 2024 data for all the countries and sorting it by the value column in descending order to see which country has the most electric vehicles in 2024
total_ev_2024 = df_total_ev[df_total_ev["time"] == "2024"].sort_values(by="value", ascending=False)
total_ev_2024

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value


In [ ]:
#pivot showing total EV per country per year
pivot_df = df_total_ev.pivot_table(index='geo', columns='time', values=['value'], aggfunc='sum')
pivot_df

value                                                           \
time    2013    2014    2015    2016    2017    2018    2019     2020   
geo                                                                     
DE     12000   19000   26000   34022  290491  424487  685801  1312898   
ES      2021   58624   67639  112986  171376  252654  372590   544914   
FR    128409  179947  254132  328493  428637  554140  706376  1049489   
NL      4161    6825  210609  245427  271785  314311  401917   526122   
PT     11356   14109   18597   25000   37436   56639   84452   119247   

                                          
time     2021     2022     2023     2024  
geo                                       
DE    2287209  3350643  4319725  5208437  
ES     817827  1126194  1535170  2024641  
FR    1652083  2279596  3088777  4039369  
NL     724986   957886  1254510  1621141  
PT     171992   239095   343965   472719

In [ ]:
#calculating the growth of EVs from 2013 to 2024 for each country by taking the difference between the two years
ev_growth = pivot_df.diff(axis=1)

ev_growth

value                                                                \
time  2013   2014    2015   2016    2017    2018    2019    2020    2021   
geo                                                                        
DE     NaN   7000    7000   8022  256469  133996  261314  627097  974311   
ES     NaN  56603    9015  45347   58390   81278  119936  172324  272913   
FR     NaN  51538   74185  74361  100144  125503  152236  343113  602594   
NL     NaN   2664  203784  34818   26358   42526   87606  124205  198864   
PT     NaN   2753    4488   6403   12436   19203   27813   34795   52745   

                               
time     2022    2023    2024  
geo                            
DE    1063434  969082  888712  
ES     308367  408976  489471  
FR     627513  809181  950592  
NL     232900  296624  366631  
PT      67103  104870  128754

In [ ]:
# Calculate the year-over-year percentage growth in electric vehicles for each country and round the results to two decimal places.
ev_growth_percent = pivot_df.pct_change(axis=1) * 100
ev_growth_percent = ev_growth_percent.round(2)

ev_growth_percent

value                                                               \
time  2013     2014     2015   2016    2017   2018   2019   2020   2021   
geo                                                                       
DE     NaN    58.33    36.84  30.85  753.83  46.13  61.56  91.44  74.21   
ES     NaN  2800.74    15.38  67.04   51.68  47.43  47.47  46.25  50.08   
FR     NaN    40.14    41.23  29.26   30.49  29.28  27.47  48.57  57.42   
NL     NaN    64.02  2985.85  16.53   10.74  15.65  27.87  30.90  37.80   
PT     NaN    24.24    31.81  34.43   49.74  51.30  49.11  41.20  44.23   

                           
time   2022   2023   2024  
geo                        
DE    46.49  28.92  20.57  
ES    37.71  36.31  31.88  
FR    37.98  35.50  30.78  
NL    32.12  30.97  29.23  
PT    39.02  43.86  37.43

In [ ]:
# Filter the 2024 dataset to include only fully electric vehicles, then create a pivot table showing the total number of electric vehicles by country.
df_2024_ev_ele = total_ev_2024[
    total_ev_2024["mot_nrg_label"] == "Electricity"
]
pivot_df_2024_ev = df_2024_ev_ele.pivot_table(index='geo', columns='time', values=['value'], aggfunc='sum')
pivot_df_2024_ev

geo


In [ ]:
# Retrieve passenger cars per 1,000 inhabitants data from the Eurostat API, convert the JSON response into a DataFrame, and display a sample of the data.
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/road_eqs_carhab?lang=en"

response = requests.get(url)
response.raise_for_status()

df_pop = response.json()

df_pop

df_carhab = jsonstat_to_df(df_pop)

df_carhab.sample(5)

,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value
1074,A,Annual,NR,Number,UK,United Kingdom,2001,2001,435
1012,A,Annual,NR,Number,NO,Norway,2008,2008,458
916,A,Annual,NR,Number,SE,Sweden,2001,2001,451
787,A,Annual,NR,Number,RO,Romania,2012,2012,224
269,A,Annual,NR,Number,EL,Greece,2001,2001,314


In [ ]:
# Display the number of records available for each country in the passenger cars per 1,000 inhabitants dataset.
df_carhab["geo"].value_counts()

geo
PL           35
CY           35
TR           35
LI           35
SE           35
FI           35
SK           35
SI           35
BE           35
AT           35
NL           35
LT           35
LV           35
HU           35
IT           35
EL           35
BG           35
DK           35
DE           35
HR           35
IE           35
RO           34
FR           34
EE           34
CH           34
MT           33
CZ           33
NO           32
LU           32
ES           31
UK           31
PT           28
MK           23
IS           22
MD           16
EU27_2020    16
AL           15
RS           15
BA           15
GE           13
ME           12
XK           12
Name: count, dtype: int64

In [ ]:
#filtering the data to only include the countries we are interested in (PT, NL, DE, FR, ES) and dropping the columns that we don't need for our analysis
df_pop2 = df_carhab[df_carhab["geo"].isin(["PT","NL","DE","FR","ES"])]
df_pop2.sample(15)

,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value
667,A,Annual,NR,Number,NL,Netherlands,2024,2024,513
657,A,Annual,NR,Number,NL,Netherlands,2014,2014,472
644,A,Annual,NR,Number,NL,Netherlands,2001,2001,417
643,A,Annual,NR,Number,NL,Netherlands,2000,2000,409
758,A,Annual,NR,Number,PT,Portugal,2017,2017,490
319,A,Annual,NR,Number,ES,Spain,2020,2020,549
663,A,Annual,NR,Number,NL,Netherlands,2020,2020,497
161,A,Annual,NR,Number,DE,Germany,1997,1997,504
163,A,Annual,NR,Number,DE,Germany,1999,1999,515
328,A,Annual,NR,Number,FR,France,1994,1994,420


In [ ]:
#calculating the number of cars per 100k people for each country by multiplying the value column by 100
df_pop3 = df_pop2.copy()
df_pop3["value_per_100k"] = df_pop3["value"] *100
df_pop3.sample(5)


,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value,value_per_100k
314,A,Annual,NR,Number,ES,Spain,2015,2015,482,48200
173,A,Annual,NR,Number,DE,Germany,2009,2009,510,51000
305,A,Annual,NR,Number,ES,Spain,2006,2006,470,47000
183,A,Annual,NR,Number,DE,Germany,2019,2019,574,57400
175,A,Annual,NR,Number,DE,Germany,2011,2011,534,53400


In [ ]:
#changing the time column to an integer to be able to filter the data by year
df_pop4= df_pop3.copy()
df_pop4["time"] = df_pop4["time"].astype(int)
df_pop4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 163 entries, 154 to 765
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   freq            163 non-null    object
 1   freq_label      163 non-null    object
 2   unit            163 non-null    object
 3   unit_label      163 non-null    object
 4   geo             163 non-null    object
 5   geo_label       163 non-null    object
 6   time            163 non-null    int64 
 7   time_label      163 non-null    object
 8   value           163 non-null    int64 
 9   value_per_100k  163 non-null    int64 
dtypes: int64(3), object(7)
memory usage: 14.0+ KB


In [ ]:
# Filter the dataset to include only records from 2013 to 2024 and verify the years remaining in the dataset.
df_pop4=df_pop4[df_pop4["time"].between(2013, 2024)]
df_pop4["time"].unique()


array([2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023,
       2024])

In [ ]:
df_pop4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 177 to 765
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   freq            60 non-null     object
 1   freq_label      60 non-null     object
 2   unit            60 non-null     object
 3   unit_label      60 non-null     object
 4   geo             60 non-null     object
 5   geo_label       60 non-null     object
 6   time            60 non-null     int64 
 7   time_label      60 non-null     object
 8   value           60 non-null     int64 
 9   value_per_100k  60 non-null     int64 
dtypes: int64(3), object(7)
memory usage: 5.2+ KB


In [ ]:
df_pop4.sample(5)


,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value,value_per_100k
319,A,Annual,NR,Number,ES,Spain,2020,2020,549,54900
180,A,Annual,NR,Number,DE,Germany,2016,2016,555,55500
761,A,Annual,NR,Number,PT,Portugal,2020,2020,535,53500
316,A,Annual,NR,Number,ES,Spain,2017,2017,529,52900
667,A,Annual,NR,Number,NL,Netherlands,2024,2024,513,51300


In [ ]:
# Create a pivot table showing the number of passenger cars per 1,000 inhabitants for each country across the selected years.
pivot_df_pop = df_pop4.pivot_table(index='geo', columns='time', values=['value_per_100k'], aggfunc='sum')
pivot_df_pop

value_per_100k                                                          \
time           2013   2014   2015   2016   2017   2018   2019   2020   2021   
geo                                                                           
DE            54300  54700  54800  55500  56100  56700  57400  58000  58300   
ES            47400  47500  48200  49200  52900  53900  54600  54900  55400   
FR            54600  55100  55800  56400  56900  56800  56900  56800  57000   
NL            47100  47200  47700  48100  48700  48900  49300  49700  50200   
PT            41400  45200  45600  46900  49000  51100  52500  53500  54100   

                           
time   2022   2023   2024  
geo                        
DE    58700  58800  59000  
ES    55300  55100  54300  
FR    56900  57300  57700  
NL    50100  50500  51300  
PT    55200  55000  57100

In [ ]:
#filtering the data to only include rows where the "mot_nrg_label" column contains the word "Total"
df_total_cars = df_total[df_total["mot_nrg_label"] == "Total"]
df_total_cars.sample(10)

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
807,A,Annual,NR,Number,TOTAL,Total,DE,Germany,2023,2023,49098685
607,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2015,2015,8100864
186,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2024,2024,6135517
184,A,Annual,NR,Number,TOTAL,Total,PT,Portugal,2022,2022,5778584
392,A,Annual,NR,Number,TOTAL,Total,FR,France,2016,2016,37673476
797,A,Annual,NR,Number,TOTAL,Total,DE,Germany,2013,2013,43851000
606,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2014,2014,7979083
609,A,Annual,NR,Number,TOTAL,Total,NL,Netherlands,2017,2017,8373244
0,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2013,2013,22025000
3,A,Annual,NR,Number,TOTAL,Total,ES,Spain,2016,2016,22876830


In [ ]:
#we need it to know hoew many eltric cars per geo to then do the % of the tot
df_total_ev.sample(10)

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
324,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,PT,Portugal,2020,2020,4837
702,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,NL,Netherlands,2024,2024,668700
892,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,DE,Germany,2020,2020,543802
710,A,Annual,NR,Number,ELC_PET_PI,Plug-in hybrid petrol-electric,NL,Netherlands,2022,2022,175720
534,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,FR,France,2014,2014,553
70,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,ES,Spain,2016,2016,105608
115,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,ES,Spain,2017,2017,45
36,A,Annual,NR,Number,ELC,Electricity,ES,Spain,2015,2015,5243
312,A,Annual,NR,Number,ELC_DIE_HYB,Hybrid diesel-electric,PT,Portugal,2019,2019,13031
926,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2022,2022,62162


In [ ]:
df_total_ev2= df_total_ev.copy()

In [ ]:
# Group the data by geographical location,  and time, then sum the values fo all types of electric vehicles to get the total number of electric vehicles per country per year
df_total_ev_grouped = (
    df_total_ev
    .groupby(["geo_label", "time"], as_index=False)["value"]
    .sum()
)
df_total_ev_grouped.head()

,geo_label,time,value
0,France,2013,128409
1,France,2014,179947
2,France,2015,254132
3,France,2016,328493
4,France,2017,428637


In [ ]:
#we double checked if germany was ok
df_total_germany = df_total_ev[df_total_ev["geo_label"] == "Germany"]
df_total_germany = df_total_germany[df_total_germany["time"] == 2017]
df_total_germany

,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,geo_label,time,time_label,value
861,A,Annual,NR,Number,ELC,Electricity,DE,Germany,2017,2017,53861
889,A,Annual,NR,Number,ELC_PET_HYB,Hybrid electric-petrol,DE,Germany,2017,2017,181885
897,A,Annual,NR,Number,ELC_PET_PI,Plug-in hybrid petrol-electric,DE,Germany,2017,2017,42569
913,A,Annual,NR,Number,ELC_DIE_HYB,Hybrid diesel-electric,DE,Germany,2017,2017,10336
921,A,Annual,NR,Number,ELC_DIE_PI,Plug-in hybrid diesel-electric,DE,Germany,2017,2017,1840


In [ ]:
# Rename the vehicle count columns to clearly distinguish total electric vehicles from total passenger cars before merging.
df_total_ev_grouped = df_total_ev_grouped.rename(
    columns={"value": "total_ev_value"}
)

df_total_cars_clean = df_total_cars.rename(
    columns={"value": "total_cars_value"}
)

In [ ]:
#merging the two dataframes together to have a complete dataset for all the countries with the total number of electric vehicles and the total number of cars per country per year
df_combined = df_total_ev_grouped.merge(
    df_total_cars_clean,
    on=["geo_label", "time"],
    how="inner"
)

df_combined.head()

,geo_label,time,total_ev_value,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,time_label,total_cars_value
0,France,2013,128409,A,Annual,NR,Number,TOTAL,Total,FR,2013,36149230
1,France,2014,179947,A,Annual,NR,Number,TOTAL,Total,FR,2014,36647883
2,France,2015,254132,A,Annual,NR,Number,TOTAL,Total,FR,2015,37180101
3,France,2016,328493,A,Annual,NR,Number,TOTAL,Total,FR,2016,37673476
4,France,2017,428637,A,Annual,NR,Number,TOTAL,Total,FR,2017,38117663


In [ ]:
#calculating the percentage of electric vehicles per total cars for each country per year by dividing the total number of electric vehicles by the total number of cars and multiplying by 100 to get the percentage
df_combined["ev_per_total_cars"] = (df_combined["total_ev_value"] / df_combined["total_cars_value"]) * 100
df_combined.sample(15)

,geo_label,time,total_ev_value,freq,freq_label,unit,unit_label,mot_nrg,mot_nrg_label,geo,time_label,total_cars_value,ev_per_total_cars
16,Germany,2017,290491,A,Annual,NR,Number,TOTAL,Total,DE,2017,46474594,0.625053
5,France,2018,554140,A,Annual,NR,Number,TOTAL,Total,FR,2018,38229305,1.449516
59,Spain,2024,2024641,A,Annual,NR,Number,TOTAL,Total,ES,2024,26700972,7.582649
29,Netherlands,2018,314311,A,Annual,NR,Number,TOTAL,Total,NL,2018,8442982,3.722749
31,Netherlands,2020,526122,A,Annual,NR,Number,TOTAL,Total,NL,2020,8686419,6.056834
56,Spain,2021,817827,A,Annual,NR,Number,TOTAL,Total,ES,2021,26293882,3.110332
54,Spain,2019,372590,A,Annual,NR,Number,TOTAL,Total,ES,2019,25836586,1.442102
13,Germany,2014,19000,A,Annual,NR,Number,TOTAL,Total,DE,2014,44403000,0.042790
14,Germany,2015,26000,A,Annual,NR,Number,TOTAL,Total,DE,2015,45071000,0.057687
7,France,2020,1049489,A,Annual,NR,Number,TOTAL,Total,FR,2020,38470122,2.728063


In [ ]:
# Remove unnecessary columns from the merged dataset to keep only the fields required for the analysis.

drop_list=["time_label","unit_label","unit","freq","freq_label","mot_nrg_label"]
df_combined_clean = df_combined.drop(columns=drop_list)
df_combined_clean.sample(10)

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars
10,France,2023,3088777,TOTAL,FR,39358421,7.847817
31,Netherlands,2020,526122,TOTAL,NL,8686419,6.056834
25,Netherlands,2014,6825,TOTAL,NL,7979083,0.085536
57,Spain,2022,1126194,TOTAL,ES,26605478,4.232940
9,France,2022,2279596,TOTAL,FR,38973339,5.849116
18,Germany,2019,685801,TOTAL,DE,47715977,1.437257
2,France,2015,254132,TOTAL,FR,37180101,0.683516
7,France,2020,1049489,TOTAL,FR,38470122,2.728063
12,Germany,2013,12000,TOTAL,DE,43851000,0.027365
50,Spain,2015,67639,TOTAL,ES,22355549,0.302560


In [ ]:
df_pop4.head()


,freq,freq_label,unit,unit_label,geo,geo_label,time,time_label,value,value_per_100k
177,A,Annual,NR,Number,DE,Germany,2013,2013,543,54300
178,A,Annual,NR,Number,DE,Germany,2014,2014,547,54700
179,A,Annual,NR,Number,DE,Germany,2015,2015,548,54800
180,A,Annual,NR,Number,DE,Germany,2016,2016,555,55500
181,A,Annual,NR,Number,DE,Germany,2017,2017,561,56100


In [ ]:
# Combine the cleaned data with population data
df_combined_all = df_combined_clean.merge(
    df_pop4,
    on=["geo", "geo_label", "time"],
    how="inner"
)
df_combined_all.sample(10)

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,value,value_per_100k
28,Netherlands,2017,271785,TOTAL,NL,8373244,3.245875,A,Annual,NR,Number,2017,487,48700
24,Netherlands,2013,4161,TOTAL,NL,7932290,0.052456,A,Annual,NR,Number,2013,471,47100
51,Spain,2016,112986,TOTAL,ES,22876830,0.493888,A,Annual,NR,Number,2016,492,49200
18,Germany,2019,685801,TOTAL,DE,47715977,1.437257,A,Annual,NR,Number,2019,574,57400
17,Germany,2018,424487,TOTAL,DE,47095784,0.901327,A,Annual,NR,Number,2018,567,56700
30,Netherlands,2019,401917,TOTAL,NL,8584391,4.681951,A,Annual,NR,Number,2019,493,49300
19,Germany,2020,1312898,TOTAL,DE,48248584,2.721112,A,Annual,NR,Number,2020,580,58000
10,France,2023,3088777,TOTAL,FR,39358421,7.847817,A,Annual,NR,Number,2023,573,57300
23,Germany,2024,5208437,TOTAL,DE,49339166,10.556394,A,Annual,NR,Number,2024,590,59000
48,Spain,2013,2021,TOTAL,ES,22025000,0.009176,A,Annual,NR,Number,2013,474,47400


In [ ]:
# Rename the vehicle ownership columns to provide clear and descriptive names for cars per 1,000 inhabitants and cars per 100,000 inhabitants.
df_combined_all= df_combined_all.rename(columns={"value": "cars_per_1k"})
df_combined_all= df_combined_all.rename(columns={"value_per_100k": "cars_per_100k"})

In [ ]:
# Pivot the data for Spain
df_spain_pivot = (
    df_combined_all[df_combined_all["geo"] == "ES"]
    .pivot_table(
        index="time",
        values=["total_ev_value", "total_cars_value","geo", "cars_per_100k","ev_per_total_cars"],
        aggfunc="sum"
    )
    .reset_index()
)

df_spain_pivot

,time,cars_per_100k,ev_per_total_cars,geo,total_cars_value,total_ev_value
0,2013,47400,0.009176,ES,22025000,2021
1,2014,47500,0.266116,ES,22029512,58624
2,2015,48200,0.302560,ES,22355549,67639
3,2016,49200,0.493888,ES,22876830,112986
4,2017,52900,0.694489,ES,24676557,171376
5,2018,53900,0.998467,ES,25304190,252654
6,2019,54600,1.442102,ES,25836586,372590
7,2020,54900,2.093058,ES,26034347,544914
8,2021,55400,3.110332,ES,26293882,817827
9,2022,55300,4.232940,ES,26605478,1126194


In [ ]:
df_combined_all.head()

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,cars_per_1k,cars_per_100k
0,France,2013,128409,TOTAL,FR,36149230,0.355219,A,Annual,NR,Number,2013,546,54600
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100
2,France,2015,254132,TOTAL,FR,37180101,0.683516,A,Annual,NR,Number,2015,558,55800
3,France,2016,328493,TOTAL,FR,37673476,0.871948,A,Annual,NR,Number,2016,564,56400
4,France,2017,428637,TOTAL,FR,38117663,1.124510,A,Annual,NR,Number,2017,569,56900


In [ ]:
# Calculate the estimated number of electric vehicles per 100,000 inhabitants using the EV share of total passenger cars and vehicle ownership data.

df_combined_all["ev_per_100k"] = (
    df_combined_all["cars_per_100k"] * df_combined_all["ev_per_total_cars"] / 100
).round(0).astype(int)

df_combined_all.head()

,geo_label,time,total_ev_value,mot_nrg,geo,total_cars_value,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,cars_per_1k,cars_per_100k,ev_per_100k
0,France,2013,128409,TOTAL,FR,36149230,0.355219,A,Annual,NR,Number,2013,546,54600,194
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100,271
2,France,2015,254132,TOTAL,FR,37180101,0.683516,A,Annual,NR,Number,2015,558,55800,381
3,France,2016,328493,TOTAL,FR,37673476,0.871948,A,Annual,NR,Number,2016,564,56400,492
4,France,2017,428637,TOTAL,FR,38117663,1.124510,A,Annual,NR,Number,2017,569,56900,640


In [ ]:
# Rename the total vehicle count columns to improve clarity and readability in the final combined dataset.
df_combined_all= df_combined_all.rename(columns={"total_cars_value": "total_cars"})
df_combined_all= df_combined_all.rename(columns={"total_ev_value": "total_ev"})
df_combined_all.head()

,geo_label,time,total_ev,mot_nrg,geo,total_cars,ev_per_total_cars,freq,freq_label,unit,unit_label,time_label,cars_per_1k,cars_per_100k,ev_per_100k
0,France,2013,128409,TOTAL,FR,36149230,0.355219,A,Annual,NR,Number,2013,546,54600,194
1,France,2014,179947,TOTAL,FR,36647883,0.491016,A,Annual,NR,Number,2014,551,55100,271
2,France,2015,254132,TOTAL,FR,37180101,0.683516,A,Annual,NR,Number,2015,558,55800,381
3,France,2016,328493,TOTAL,FR,37673476,0.871948,A,Annual,NR,Number,2016,564,56400,492
4,France,2017,428637,TOTAL,FR,38117663,1.124510,A,Annual,NR,Number,2017,569,56900,640


In [ ]:
# Create a yearly summary table for Spain showing total electric vehicles, total passenger cars, vehicle ownership, and EV adoption metrics.

df_spain_pivot = (
    df_combined_all[df_combined_all["geo"] == "ES"]
    .pivot_table(
        index="time",
        values=["total_ev", "total_cars","geo", "cars_per_100k","ev_per_total_cars","ev_per_100k"],
        aggfunc="sum"
    )
    .reset_index()
)

df_spain_pivot

,time,cars_per_100k,ev_per_100k,ev_per_total_cars,geo,total_cars,total_ev
0,2013,47400,4,0.009176,ES,22025000,2021
1,2014,47500,126,0.266116,ES,22029512,58624
2,2015,48200,146,0.302560,ES,22355549,67639
3,2016,49200,243,0.493888,ES,22876830,112986
4,2017,52900,367,0.694489,ES,24676557,171376
5,2018,53900,538,0.998467,ES,25304190,252654
6,2019,54600,787,1.442102,ES,25836586,372590
7,2020,54900,1149,2.093058,ES,26034347,544914
8,2021,55400,1723,3.110332,ES,26293882,817827
9,2022,55300,2341,4.232940,ES,26605478,1126194


In [ ]:
# Group the combined dataset by country and year to summarize EV adoption, total electric vehicles, and total passenger cars.
df_all_geo_grouped = (
    df_combined_all
    .groupby(["geo_label", "time"], as_index=False)[["ev_per_100k", "total_ev", "total_cars"]]
    .sum()
)
df_all_geo_grouped

,geo_label,time,ev_per_100k,total_ev,total_cars
0,France,2013,194,128409,36149230
1,France,2014,271,179947,36647883
2,France,2015,381,254132,37180101
3,France,2016,492,328493,37673476
4,France,2017,640,428637,38117663
5,France,2018,823,554140,38229305
6,France,2019,1046,706376,38423556
7,France,2020,1550,1049489,38470122
8,France,2021,2426,1652083,38818914
9,France,2022,3328,2279596,38973339


In [ ]:
# Calculate the growth rate of ev_per_100k for each geo_label
df_all_geo_grouped["ev_per_100k_growth_rate"] = (
    df_all_geo_grouped
    .groupby("geo_label")["ev_per_100k"]
    .pct_change() * 100
)

In [ ]:
df_all_geo_grouped

,geo_label,time,ev_per_100k,total_ev,total_cars,ev_per_100k_growth_rate
0,France,2013,194,128409,36149230,NaN
1,France,2014,271,179947,36647883,39.690722
2,France,2015,381,254132,37180101,40.590406
3,France,2016,492,328493,37673476,29.133858
4,France,2017,640,428637,38117663,30.081301
5,France,2018,823,554140,38229305,28.593750
6,France,2019,1046,706376,38423556,27.095990
7,France,2020,1550,1049489,38470122,48.183556
8,France,2021,2426,1652083,38818914,56.516129
9,France,2022,3328,2279596,38973339,37.180544


In [ ]:
# Filter the dataset to include only records from 2017 onwards for the comparative EV adoption analysis.
df_all_geo_grouped = df_all_geo_grouped[
    df_all_geo_grouped["time"].astype(int) >= 2017
]

In [ ]:
df_all_geo_grouped["ev_per_100k_growth_rate"] = df_all_geo_grouped["ev_per_100k_growth_rate"].round(2)

In [ ]:
df_all_geo_grouped

,geo_label,time,ev_per_100k,total_ev,total_cars,ev_per_100k_growth_rate
4,France,2017,640,428637,38117663,30.08
5,France,2018,823,554140,38229305,28.59
6,France,2019,1046,706376,38423556,27.10
7,France,2020,1550,1049489,38470122,48.18
8,France,2021,2426,1652083,38818914,56.52
9,France,2022,3328,2279596,38973339,37.18
10,France,2023,4497,3088777,39358421,35.13
11,France,2024,5865,4039369,39738933,30.42
16,Germany,2017,351,290491,46474594,756.10
17,Germany,2018,511,424487,47095784,45.58


## WEBSCARPING A NEWS ARTICLE

In [ ]:
# 1- Pick the URL and save it
news_url = "https://www.globalbankingandfinance.com/electriccars-spain-one/"

# 2- Identify ourselves as a browser (tested and working!) = HEADERS
headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"}
# "I am Chrome 116, running on Windows 10 64-bit, using WebKit as my rendering engine."

# 3- Make the request, passing the URL and the HEADERS
news_response = requests.get(news_url, headers=headers)

# 4- Check the status (Good Practice!)
if news_response.status_code == 200:
    print("Connection successful!")
else:
    print(f"Connection failed: {news_response.status_code}")

Connection successful!


In [ ]:
#printing the status code and first 1000 characters of the articile
print(news_response.status_code)
print(news_response.text[:1000])

200
<!DOCTYPE html><html lang="en"> <head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=6, viewport-fit=cover"><link rel="preload" as="image" href="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format" imagesrcset="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=400&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format 400w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=640&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format 640w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&#38;q=72&#38;fm=webp&#38;fit=crop&#38;auto=format 720w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=1024

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
#Parcing the response
news_soup = BeautifulSoup(news_response.content, "html.parser")
print(news_soup.prettify()[:2000])

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=6, viewport-fit=cover" name="viewport"/>
  <link as="image" fetchpriority="high" href="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format" imagesizes="(max-width: 768px) 100vw, (max-width: 1024px) 90vw, (max-width: 1536px) 720px, 660px" imagesrcset="https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=400&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format 400w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=640&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format 640w, https://cdn.sanity.io/images/v0gkry1w/production/e1d03ab7c40a2eeba422bc1806547c8587db3e3b-640x430.webp?w=720&amp;q=72&amp;fm=webp&amp;fit=crop&amp;auto=format 7

In [ ]:
# A quick shortcut: soup.title gives us the <title> tag directly
print(news_soup.title)                   # Full tag: <title>Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking &amp; Finance Review</title>
print(news_soup.title.string)            # Just the text: Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking & Finance Review

<title>Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking &amp; Finance Review</title>
Spain's $1.5 Billion EV Boost - Electric Vehicle Market Growth | Global Banking & Finance Review


In [ ]:
# Extract all paragraph elements from the news article and display the total number of paragraphs found.
news_para = news_soup.find_all("p")
print(len(news_para))

23


In [ ]:
# Preview the first 13 paragraphs extracted from the news article to inspect the scraped content.
news_para[0:13]

[<p class="px-4 py-2 border border-gray-200 mt-2 text-sm leading-relaxed text-gray-700" data-astro-cid-qusqgt37=""> Global Banking &amp; Finance Review® is an online platform offering news, analysis, and opinion on the latest trends, developments, and innovations in the banking and finance industry worldwide. The platform covers a diverse range of topics, including banking, insurance, investment, wealth management, fintech, and regulatory issues. The website publishes news, press releases, opinion and advertorials on various financial organizations, products and services which are commissioned from various Companies, Organizations, PR agencies, Bloggers etc. These commissioned articles are commercial in nature. This is not to be considered as financial advice and should be considered only for information purposes. It does not reflect the views or opinion of our website and is not to be considered an endorsement or a recommendation. We cannot guarantee the accuracy or applicability of a

In [ ]:
# Exclude the introductory paragraphs and keep only the main article content for further text extraction and analysis.
article_paragraphs = news_para[5:]
print(article_paragraphs)

[<p>MADRID, Dec 3 (Reuters) - Spain will provide nearly 1.3 billion euros ($1.52 billion) to support its electric vehicle market and industry next year as part of a plan to lift the share of EVs produced in the country to 95% by 2035, Prime Minister Pedro Sanchez said on Wednesday.</p>, <p>In the first 10 months of 2025, the share of fully electric and plug-in hybrid vehicles made in Spain totalled around 10%, industry data show. Self-charging hybrids accounted for 26.7%.</p>, <p>Around 20% of vehicles across the EU last year were fully electric or plug-in hybrids.</p>, <p>Spain's plan includes 400 million euros in direct subsidies in 2026 for consumers to buy EVs and another 580 million euros under the country's EU-funded scheme supporting industrial investment.</p>, <p>It will also add 300 million euros to install charging points along roads still lacking coverage.</p>, <p>Spain is stepping up support for its automotive sector as Chinese EV brands like BYD rapidly expand, undercuttin

In [ ]:
# Extract the text from each paragraph, store it in a list, and display the extracted article content.
article_text_list = []

for p in article_paragraphs:
    text = p.get_text(strip=True)
    article_text_list.append(text)

print(article_text_list)

['MADRID, Dec 3 (Reuters) - Spain will provide nearly 1.3 billion euros ($1.52 billion) to support its electric vehicle market and industry next year as part of a plan to lift the share of EVs produced in the country to 95% by 2035, Prime Minister Pedro Sanchez said on Wednesday.', 'In the first 10 months of 2025, the share of fully electric and plug-in hybrid vehicles made in Spain totalled around 10%, industry data show. Self-charging hybrids accounted for 26.7%.', 'Around 20% of vehicles across the EU last year were fully electric or plug-in hybrids.', "Spain's plan includes 400 million euros in direct subsidies in 2026 for consumers to buy EVs and another 580 million euros under the country's EU-funded scheme supporting industrial investment.", 'It will also add 300 million euros to install charging points along roads still lacking coverage.', "Spain is stepping up support for its automotive sector as Chinese EV brands like BYD rapidly expand, undercutting European rivals and explo

In [ ]:
# Combine all extracted paragraphs into a single text string, separating each paragraph with a blank line for readability.

article_text = "\n\n".join(article_text_list)

In [ ]:
# Display the complete extracted text from the news article.
print(article_text)

MADRID, Dec 3 (Reuters) - Spain will provide nearly 1.3 billion euros ($1.52 billion) to support its electric vehicle market and industry next year as part of a plan to lift the share of EVs produced in the country to 95% by 2035, Prime Minister Pedro Sanchez said on Wednesday.

In the first 10 months of 2025, the share of fully electric and plug-in hybrid vehicles made in Spain totalled around 10%, industry data show. Self-charging hybrids accounted for 26.7%.

Around 20% of vehicles across the EU last year were fully electric or plug-in hybrids.

Spain's plan includes 400 million euros in direct subsidies in 2026 for consumers to buy EVs and another 580 million euros under the country's EU-funded scheme supporting industrial investment.

It will also add 300 million euros to install charging points along roads still lacking coverage.

Spain is stepping up support for its automotive sector as Chinese EV brands like BYD rapidly expand, undercutting European rivals and exploiting the co

In [ ]:
# Save the extracted news article as a text file for future analysis and reference.
with open("spain_ev_news_article.txt", "w", encoding="utf-8") as file:
    file.write(article_text)